# CIC6314 — Career Recommendation System
## Section 1: A* Search Algorithm (Member 1)

**Function delivered:**
```python
find_career_path(user_profile, target_career) → list[str]
```


---
## Overview — Where A* Fits in the Pipeline

The full system runs in three stages:

1. **Rules Engine (Member 2)** — looks at the user's profile and filters out careers they clearly don't qualify for. Returns a shortlist.
2. **A* Search (this module)** — takes that shortlist and the user's target career, then finds the most efficient career progression path to get there.
3. **ML Model (Member 3)** — ranks careers by predicted fit based on patterns learned from the dataset.

A* sits in the middle — it bridges the rules-based filtering and the ML prediction by giving the user a concrete, step-by-step roadmap rather than just a recommendation.


---
## 1. Imports and Setup

`heapq` handles the priority queue. `networkx` and `matplotlib` are used for visualising the career graph. Everything else comes from `src/constants.py`, which the whole team shares — no values are hardcoded in this file.


In [ ]:
import heapq
import sys
import os
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from constants import (
    CAREER_CATEGORIES,
    EDUCATION_LEVELS,
    ALL_SKILLS,
    SAMPLE_PROFILES,
    build_user_profile,
)

EDUCATION_RANK = {level: i for i, level in enumerate(EDUCATION_LEVELS)}

print("Constants loaded.")
print(f"Careers tracked : {len(CAREER_CATEGORIES)}")
print(f"Skills tracked  : {len(ALL_SKILLS)}")
print(f"Education ranks : {EDUCATION_RANK}")


---
## 2. AI Problem Formulation

Before writing any code, the problem needs to be defined in terms A* can work with.

| Component | Definition |
|---|---|
| **State** | A career node in the graph (one of the 12 career categories) |
| **Initial state** | The career that best matches the user's current skills and education level |
| **Goal state** | The target career the user wants to reach |
| **Actions** | Move to any directly connected neighbouring career in the graph |
| **Step cost** | Number of skills required by the next career that the user doesn't currently have |
| **Path cost** | Total skill gaps accumulated across all steps in the path |
| **Solution** | A sequence of career nodes from the initial state to the goal state with minimum total skill gap |

The objective is to find the **optimal path** — the one that requires the fewest total skill acquisitions to get from where the user is now to where they want to be.

This is a well-defined single-agent search problem. A* is appropriate here because:
- The state space is small and finite (12 nodes)
- We have a meaningful heuristic (skill gap to target)
- We want the optimal solution, not just any solution


---
## 3. Career Graph

The graph is the search space. Each career is a node; edges represent realistic transitions between roles.

Edge direction matters — not every career can transition to every other. The graph covers both upward progression (e.g. Clerk → Business Analyst) and lateral moves (e.g. Software Engineer ↔ ML Engineer) since people don't always move in a straight line.

Each node stores three things:
- `skills_required` — the skills expected for someone in that role
- `min_education` — the minimum qualification needed to enter it
- `neighbors` — which careers this role can realistically transition into


In [ ]:
CAREER_GRAPH = {
    "Clerk": {
        "skills_required": ["MS Office", "Communication"],
        "min_education":   "Matric",
        "neighbors":       ["Data Entry Operator", "Sales Assistant",
                            "Junior Accountant", "School Counselor"],
    },
    "Data Entry Operator": {
        "skills_required": ["MS Office", "Accounting"],
        "min_education":   "Matric",
        "neighbors":       ["Clerk", "Junior Accountant", "Business Analyst"],
    },
    "Sales Assistant": {
        "skills_required": ["Communication", "Marketing", "MS Office"],
        "min_education":   "Intermediate",
        "neighbors":       ["Marketing Executive", "Business Analyst", "School Counselor"],
    },
    "Junior Accountant": {
        "skills_required": ["Accounting", "Financial Analysis", "MS Office"],
        "min_education":   "Intermediate",
        "neighbors":       ["Financial Analyst", "Business Analyst"],
    },
    "Marketing Executive": {
        "skills_required": ["Marketing", "Communication", "Data Analysis"],
        "min_education":   "Bachelor's",
        "neighbors":       ["Business Analyst", "Research Scientist"],
    },
    "Financial Analyst": {
        "skills_required": ["Financial Analysis", "Accounting", "Data Analysis", "SQL"],
        "min_education":   "Bachelor's",
        "neighbors":       ["Business Analyst", "Research Scientist", "Professor"],
    },
    "Business Analyst": {
        "skills_required": ["Data Analysis", "Communication", "SQL", "MS Office"],
        "min_education":   "Bachelor's",
        "neighbors":       ["ML Engineer", "Research Scientist", "Software Engineer"],
    },
    "School Counselor": {
        "skills_required": ["Counseling", "Communication"],
        "min_education":   "Bachelor's",
        "neighbors":       ["Professor", "Research Scientist"],
    },
    "Software Engineer": {
        "skills_required": ["Python", "SQL", "Data Analysis"],
        "min_education":   "Bachelor's",
        "neighbors":       ["ML Engineer", "Research Scientist", "Business Analyst"],
    },
    "ML Engineer": {
        "skills_required": ["Python", "Machine Learning", "Data Analysis", "SQL"],
        "min_education":   "Bachelor's",
        "neighbors":       ["Research Scientist", "Professor", "Software Engineer"],
    },
    "Research Scientist": {
        "skills_required": ["Python", "Machine Learning", "Data Analysis", "Financial Analysis"],
        "min_education":   "Master's",
        "neighbors":       ["Professor", "ML Engineer"],
    },
    "Professor": {
        "skills_required": ["Counseling", "Communication", "Data Analysis"],
        "min_education":   "PhD",
        "neighbors":       ["Research Scientist"],
    },
}

print(f"Graph loaded: {len(CAREER_GRAPH)} career nodes")


---
## 4. Career Graph Visualisation

The diagram below maps the full search space. Node colour shows the minimum education level required. Arrows show which transitions are possible and in which direction.


In [ ]:
edu_colours = {
    "Matric":       "#f4a261",
    "Intermediate": "#e9c46a",
    "Bachelor's":   "#57cc99",
    "Master's":     "#4cc9f0",
    "PhD":          "#9b5de5",
}

G = nx.DiGraph()
for career, info in CAREER_GRAPH.items():
    G.add_node(career)
    for neighbor in info["neighbors"]:
        G.add_edge(career, neighbor)

node_colours = [edu_colours[CAREER_GRAPH[n]["min_education"]] for n in G.nodes()]

pos = {
    "Clerk":              (0, 4),
    "Data Entry Operator":(2, 4),
    "Sales Assistant":    (0, 3),
    "Junior Accountant":  (2, 3),
    "School Counselor":   (4, 3),
    "Marketing Executive":(0, 2),
    "Financial Analyst":  (2, 2),
    "Business Analyst":   (4, 2),
    "Software Engineer":  (1, 1),
    "ML Engineer":        (3, 1),
    "Research Scientist": (5, 1),
    "Professor":          (4, 0),
}

fig, ax = plt.subplots(figsize=(14, 8))
nx.draw_networkx(
    G, pos=pos, ax=ax,
    node_color=node_colours,
    node_size=2200,
    font_size=7,
    font_weight="bold",
    arrows=True,
    arrowsize=18,
    edge_color="#555555",
    connectionstyle="arc3,rad=0.08",
)

legend_handles = [
    mpatches.Patch(color=c, label=lvl)
    for lvl, c in edu_colours.items()
]
ax.legend(handles=legend_handles, title="Min. Education", loc="lower left", fontsize=9)
ax.set_title("Career Progression Graph — A* Search Space", fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("career_graph.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph saved as career_graph.png")


---
## 5. Heuristic Design and Admissibility Proof

### What is the heuristic?

The A* heuristic **h(n)** estimates how far the current node is from the goal — without actually traversing the remaining path.

We define it as:

> **h(n) = number of skills required by the target career that the user does not currently have**

For example, if the target is *ML Engineer* (requires Python, Machine Learning, Data Analysis, SQL) and the user only has Python and SQL — h(n) = 2.

---

### Admissibility Proof

A heuristic is **admissible** if it never overestimates the true remaining cost. This is the condition that guarantees A* returns the optimal path.

**Proof:**
- Reaching the target career requires the user to eventually have all of its required skills.
- The user currently lacks some of those skills — that gap is exactly what h(n) counts.
- h(n) counts the skills missing *right now*. The actual path may require picking up additional intermediate skills along the way, making the real cost ≥ h(n).
- Therefore **h(n) ≤ true remaining cost** for every node n. ✓
- Since the heuristic never overestimates → it is admissible → A* is guaranteed to find the optimal path.

---

### Why skill gap and not something else?

A constant heuristic (h = 0) degrades A* to Dijkstra's — it works but explores more nodes than necessary. An overestimating heuristic breaks optimality. Skill gap sits in between: it's always a valid lower bound, it's easy to compute, and it's directly meaningful to the problem — which is exactly what a good heuristic should be.


---
## 6. Helper Functions

Four functions do the groundwork before A* runs.

- **`skill_gap()`** — the base operation. Takes two skill lists, returns how many skills from the second list are missing from the first. Everything else calls this.
- **`heuristic()`** — calls `skill_gap()` against the *target* career's requirements. This is h(n).
- **`edge_cost()`** — calls `skill_gap()` against the *next node's* requirements. This is the g(n) step cost when expanding a neighbour.
- **`get_start_career()`** — determines the A* starting node. It filters careers by education eligibility, then scores each remaining career by skill overlap minus skill gap. The best-scoring career is where the search begins.


In [ ]:
def skill_gap(user_skills, required_skills):
    """
    Count how many skills from required_skills the user is missing.
    Returns 0 if the user has everything needed.
    """
    return len(set(required_skills) - set(user_skills))


def heuristic(current_career, target_career, user_skills):
    """
    h(n): estimated cost from current career to target.
    = skills required by target that the user doesn't currently have.
    Admissible — never overestimates the true remaining cost.
    """
    target_skills = CAREER_GRAPH[target_career]["skills_required"]
    return skill_gap(user_skills, target_skills)


def edge_cost(to_career, user_skills):
    """
    Step cost to enter a career node.
    = skills required by that career that the user doesn't have yet.
    Moving into a role you're already qualified for costs 0.
    """
    return skill_gap(user_skills, CAREER_GRAPH[to_career]["skills_required"])


def get_start_career(user_profile):
    """
    Find the career that best matches the user's current profile.
    Used as the A* starting node.

    Filters out careers the user is educationally underqualified for,
    then scores the rest by: skill_overlap - skill_gap.
    Returns the highest-scoring career.
    """
    user_skills   = set(user_profile["skills"])
    user_edu_rank = EDUCATION_RANK.get(user_profile["education_level"], 0)
    best_career, best_score = None, float("-inf")

    for career, info in CAREER_GRAPH.items():
        if user_edu_rank < EDUCATION_RANK.get(info["min_education"], 0):
            continue
        required = set(info["skills_required"])
        score    = len(user_skills & required) - len(required - user_skills)
        if score > best_score:
            best_score, best_career = score, career

    return best_career if best_career else "Clerk"


# Sanity check
test_user = ["Python", "SQL"]
print(f"skill_gap(['Python','SQL'] vs ML Engineer) = {skill_gap(test_user, CAREER_GRAPH['ML Engineer']['skills_required'])}")
print(f"heuristic at Software Engineer, target ML Engineer = {heuristic('Software Engineer', 'ML Engineer', test_user)}")
print(f"edge_cost to ML Engineer = {edge_cost('ML Engineer', test_user)}")


---
## 7. A* Search — `find_career_path()`

### How the algorithm runs

1. Put the start node on a **priority queue** (min-heap) with priority f = g + h.
2. Pop the node with the lowest f score.
3. If it's the goal — return the path and stop.
4. Otherwise, expand its neighbours: for each unvisited neighbour, calculate its new g, h, and f scores, then push onto the queue.
5. Mark the current node as visited so it's never re-expanded — once we've found the cheapest way to reach a node, we don't need to reconsider it.
6. Repeat until the goal is found or the queue is empty.

The priority queue is the key difference from BFS. BFS explores nodes in order of distance (number of steps). A* explores in order of *estimated total cost* (f = g + h), so it steers toward the goal rather than spreading outward in all directions.

**f(n) = g(n) + h(n)**
| Term | Meaning |
|---|---|
| g(n) | Total skill gaps accumulated to reach node n |
| h(n) | Estimated remaining skill gaps from n to target |
| f(n) | Total estimated path cost — the priority queue sorts by this |


In [ ]:
def find_career_path(user_profile, target_career):
    """
    A* search — finds the optimal career progression path.

    Parameters
    ----------
    user_profile  : dict — from build_user_profile()
    target_career : str  — one of CAREER_CATEGORIES

    Returns
    -------
    list[str] — ordered career steps from current best-fit to target.
                Includes both start and target in the list.
                Returns empty list if no path exists.
    """
    if target_career not in CAREER_GRAPH:
        return []

    start       = get_start_career(user_profile)
    user_skills = user_profile["skills"]

    if start == target_career:
        return [start]

    # (f_score, g_score, current_node, path_so_far)
    # g_score is included as tiebreaker so heapq never compares strings
    start_h  = heuristic(start, target_career, user_skills)
    open_set = [(start_h, 0, start, [start])]
    visited  = set()

    while open_set:
        f, g, current, path = heapq.heappop(open_set)

        if current in visited:
            continue
        visited.add(current)

        if current == target_career:
            return path

        for neighbor in CAREER_GRAPH[current]["neighbors"]:
            if neighbor in visited:
                continue
            new_g = g + edge_cost(neighbor, user_skills)
            new_h = heuristic(neighbor, target_career, user_skills)
            heapq.heappush(open_set, (new_g + new_h, new_g, neighbor, path + [neighbor]))

    return []


---
## 8. Detailed Path Display

The output below shows each step in the path with a full skill breakdown — which skills the user already has for that role, and which ones are still missing. This makes the result actionable: the user knows exactly what to work on at each stage.


In [ ]:
def display_path_detailed(profile_name, user_profile, path):
    """Print a step-by-step career path with skill gap analysis at each stage."""
    target      = user_profile.get("target_career", "N/A")
    user_skills = set(user_profile["skills"])

    print("=" * 62)
    print(f"  Profile    : {profile_name}")
    print(f"  Education  : {user_profile['education_level']}")
    print(f"  Skills     : {sorted(user_profile['skills'])}")
    print(f"  Target     : {target}")
    print("=" * 62)

    if not path:
        print("  No path found.")
        print()
        return

    total_gap = 0
    for i, career in enumerate(path):
        required = set(CAREER_GRAPH[career]["skills_required"])
        has      = user_skills & required
        missing  = required - user_skills
        gap      = len(missing)
        total_gap += gap

        label = "★ TARGET" if career == target else f"Step {i + 1}"
        print(f"\n  [{label}] {career}")
        print(f"    Required skills : {sorted(required)}")
        print(f"    You have        : {sorted(has) if has else 'none'}")
        print(f"    Missing         : {sorted(missing) if missing else 'none'}")
        print(f"    Skill gap       : {gap}  {'✓ ready' if gap == 0 else '✗ need to acquire these'}")

    print(f"\n  Path        : {' → '.join(path)}")
    print(f"  Total steps : {len(path)}")
    print(f"  Total gaps  : {total_gap} skill(s) to acquire along this path")
    print()


---
## 9. Test Results — All Sample Profiles


In [ ]:
for name, profile in SAMPLE_PROFILES.items():
    target = profile.get("target_career")
    if not target:
        print(f"[{name}] No target_career set — skipping.\n")
        continue
    path = find_career_path(profile, target)
    display_path_detailed(name, profile, path)


---
## 10. Results Discussion

Running A* against the five sample profiles gives a few useful observations:

**Profiles already at their target** (`cs_ml_student`, `business_analyst`) return a single-node path with zero skill gap. This is correct — the algorithm recognises they're already a strong match for their goal and doesn't force unnecessary steps.

**Short progression paths** (`finance_graduate`, `psychology_grad`, `software_engineer`) return 2-step paths. In each case, the intermediate career acts as a natural stepping stone — the skills required to get there partially cover what the target needs too.

**Skill gaps in the path don't always land at the target.** For `finance_graduate`, the gaps are at *Financial Analyst* (Data Analysis and SQL), not at Junior Accountant. This is expected — A* finds the cheapest total path, and the start node was already a strong skill match for Junior Accountant.

**The heuristic guides the search efficiently.** In a graph of 12 nodes, A* typically visits 3–5 nodes before finding the goal, compared to BFS which would expand all reachable nodes in order. The skill gap estimate keeps the search focused on paths heading toward the right skill set.

**Limitation:** The career graph is manually constructed and reflects assumed career transitions. In a production system, edge weights and graph structure would ideally be derived from real-world career progression data rather than set by hand.


---
## 11. Path Visualisation

The graph below highlights the A* path for a selected profile. The target career is shown in red; intermediate steps in yellow. Change `selected_profile` to visualise a different one.


In [ ]:
selected_profile = "finance_graduate"

profile = SAMPLE_PROFILES[selected_profile]
target  = profile["target_career"]
path    = find_career_path(profile, target)

print(f"Path for '{selected_profile}': {' → '.join(path)}")

path_set     = set(path)
node_colours = []
for n in G.nodes():
    if n == target:
        node_colours.append("#e63946")
    elif n in path_set:
        node_colours.append("#f4d35e")
    else:
        node_colours.append(edu_colours[CAREER_GRAPH[n]["min_education"]])

path_edges   = [(path[i], path[i+1]) for i in range(len(path)-1)]
edge_colours = ["#e63946" if e in path_edges else "#cccccc" for e in G.edges()]
edge_widths  = [3.0 if e in path_edges else 1.0 for e in G.edges()]

fig, ax = plt.subplots(figsize=(14, 8))
nx.draw_networkx(
    G, pos=pos, ax=ax,
    node_color=node_colours,
    node_size=2200,
    font_size=7,
    font_weight="bold",
    arrows=True,
    arrowsize=18,
    edge_color=edge_colours,
    width=edge_widths,
    connectionstyle="arc3,rad=0.08",
)

legend_handles = [
    mpatches.Patch(color="#e63946", label="Target career"),
    mpatches.Patch(color="#f4d35e", label="Intermediate steps"),
    mpatches.Patch(color="#aaaaaa", label="Other careers"),
]
ax.legend(handles=legend_handles, loc="lower left", fontsize=9)
ax.set_title(f"A* Path: {' → '.join(path)}", fontsize=12, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("career_path_highlighted.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 12. Custom Profile Test

Change the values below and re-run the cell to test any user profile.


In [ ]:
custom_profile = build_user_profile(
    education_level = "Bachelor's",
    specialization  = "Computer Science",
    cgpa            = 78,
    skills          = ["Python", "SQL"],
    certifications  = "AWS Certified",
    target_career   = "ML Engineer",
)

path = find_career_path(custom_profile, custom_profile["target_career"])
display_path_detailed("custom_profile", custom_profile, path)
